# Carrying Out Error Analysis

## 1. Core Idea

**Error analysis** is the process of manually examining examples that a model gets wrong in order to determine which types of errors contribute most to the total development-set error.

Its main purpose is to answer:

> What should the team work on next?

Instead of relying only on intuition, we inspect actual model failures and use evidence to prioritize engineering effort.

## 2. Why Error Analysis Is Necessary

After building an initial system, there may be many possible improvements:

* collect more data
* improve preprocessing
* increase model capacity
* handle blurry inputs
* fix incorrect labels
* add a specialized component

It is usually impossible to pursue all of them at once.

Error analysis helps estimate which problems are large enough to justify attention.

## 3. Basic Procedure

A practical error-analysis process is:

1. Select a random sample of incorrectly classified examples from the dev set.
2. Inspect each example manually.
3. Create meaningful error categories.
4. Mark which categories apply to each example.
5. Calculate the percentage of errors belonging to each category.
6. Estimate the maximum possible improvement from fixing each category.
7. Prioritize the most valuable and feasible direction.

A sample of around 50–100 errors is often enough to reveal major patterns.

## 4. Error-Analysis Table

A simple table may look like this:

| Example    | Dog | Great Cat | Blurry Image | Filtered Image |
| ---------- | --: | --------: | -----------: | -------------: |
| 1          |   ✓ |           |              |                |
| 2          |     |         ✓ |            ✓ |                |
| 3          |     |           |            ✓ |              ✓ |
| ...        | ... |       ... |          ... |            ... |
| Percentage |  8% |       43% |          12% |            12% |

Each row represents one incorrectly classified example.

Each column represents a possible error category.

An example may belong to more than one category. Therefore, the percentages do not have to sum to 100%.

## 5. Ceiling Analysis

The percentage of errors in a category provides an estimate of its **maximum possible impact**.

Suppose:

* the current error rate is 10%
* dog images account for 5% of all errors

Even if every dog-related error were fixed perfectly, the maximum absolute reduction would be:

$$
10% \times 5% = 0.5%
$$

The error rate could improve at most from:

$$
10% \rightarrow 9.5%
$$

In general, if:

* $E$ is the current error rate
* $p_c$ is the proportion of errors in category $c$

then the maximum absolute improvement is approximately:

$$
\Delta E_{\max,c} = E \times p_c
$$

This is an ideal upper bound. Real improvement is usually smaller.

## 6. How to Use the Results

A category with a very small ceiling is usually not worth a large engineering investment.

A category with a large ceiling may be promising, but priority should also consider:

* implementation difficulty
* data-collection cost
* engineering time
* expected probability of success

Therefore, the largest category is not automatically the first one to fix.

## 7. Build the First System Quickly

A key lesson is:

> Build a reasonable first system quickly, then use error analysis to decide what to improve.

Before a baseline exists, the team does not know which failure modes are actually important.

The recommended workflow is:

$$
\text{Build}
\rightarrow
\text{Evaluate}
\rightarrow
\text{Analyze Errors}
\rightarrow
\text{Choose the Next Improvement}
$$

The first system does not need to be perfect, but it must be correct enough for its errors to be meaningful.

## 8. Important Cautions

### Do Not Rely Only on Intuition

A failure mode may appear important but account for only a small percentage of total errors.

### Use a Representative Sample

Do not inspect only unusual or memorable failures. Random sampling gives a more reliable picture.

### Use Actionable Categories

Categories such as “hard example” or “model confused” are too vague.

Useful categories should suggest possible interventions.

### Use the Dev Set

Error analysis should be performed on the dev set, not the test set.

If test-set errors are repeatedly inspected and used to improve the model, the test set becomes part of the development process.

### Association Is Not Proof of Cause

If many errors occur on blurry images, this suggests that blur may be important, but it does not prove that blur is the direct cause.

Error analysis generates hypotheses. Experiments are still needed to verify them.

## 9. Multi-Stage Systems

For a system with several components, error analysis should identify where the failure occurs.

For example, in a RAG system:

$$
\text{Retriever}
\rightarrow
\text{Reranker}
\rightarrow
\text{LLM}
$$

A wrong answer may result from:

* retrieval failure
* reranking failure
* generation failure
* incorrect citation formatting

Without module-level analysis, the team may improve the wrong component.

## 10. Essence

The essence of carrying out error analysis is:

> Examine real development-set errors, organize them into useful categories, estimate the potential gain from fixing each category, and use that evidence to choose the next engineering priority.

The central principle is:

> Do not spend significant effort fixing a type of error before measuring how much it contributes to the total error.


# Cleaning Up Incorrectly Labeled Data

## 1. Core Idea

Real-world datasets often contain incorrectly labeled examples.

The important question is not:

> Are there any incorrect labels?

The important question is:

> Are incorrect labels frequent or harmful enough to justify cleaning them?

Cleaning labels can be expensive. Therefore, the decision should be based on their impact on training and evaluation.

## 2. Incorrect Labels in the Training Set

Deep neural networks can often tolerate a small amount of incorrect training labels, especially when:

- the training set is large
- labeling errors are relatively random
- most examples are labeled correctly
- errors are not concentrated in an important class

Therefore, it is usually unnecessary to manually inspect and correct every training example.

However, training-label errors become more dangerous when they are systematic.

Examples include:

- all examples from one source are assigned the wrong label
- one class is consistently confused with another
- an annotator repeatedly misinterprets the labeling guideline
- rare positive examples are frequently labeled as negative

Systematic noise can cause the model to learn an incorrect pattern.

## 3. Incorrect Labels in the Dev and Test Sets

Incorrect labels in the dev and test sets are especially important because these datasets control evaluation.

The dev set is used to:

- compare models
- tune hyperparameters
- choose engineering priorities

The test set is used to estimate final generalization performance.

Incorrect evaluation labels can cause the team to:

- select the wrong model
- miscalculate performance
- investigate failure modes that are not real
- make incorrect project decisions

## 4. Measuring the Importance of Incorrect Labels

During error analysis, add a category for incorrectly labeled examples.

Suppose:

- the current dev error is 10%
- incorrect labels account for 6% of the inspected errors

The measured error caused by incorrect labels is approximately:

$$
10\% \times 6\% = 0.6\%
$$

If all such labels were corrected, the measured dev error could improve at most from:

$$
10\% \rightarrow 9.4\%
$$

If the effect is very small and does not change model selection, cleaning may not be worth the cost.

If incorrect labels account for a large percentage of dev errors, the evaluation set should be reviewed more carefully.

## 5. Keep Dev and Test Processing Consistent

If labels are corrected in the dev set, the same labeling rules and review process should be applied to the test set.

The dev and test sets should have:

- the same label definitions
- similar annotation quality
- the same treatment of ambiguous examples
- consistent correction procedures

Cleaning only the dev set may create an artificial difference between dev and test evaluation.

## 6. Review Both Correct and Incorrect Predictions

Error analysis usually examines examples that the model predicts incorrectly.

However, if the team decides to clean the evaluation set formally, it should not review only those examples.

Consider an example where:

- the true class is cat
- the dataset label is not cat
- the model predicts not cat

The model prediction matches the incorrect dataset label, so the example appears to be correct even though both the label and prediction are wrong.

Therefore:

> Formal dev/test cleaning should examine both apparently correct and apparently incorrect predictions.

Otherwise, the corrected evaluation set may become biased toward the current model.

## 7. Incorrect Labels Versus Ambiguous Examples

Not every disagreement indicates an incorrect label.

An example may be genuinely ambiguous because:

- the input is unclear
- important information is missing
- multiple labels may be reasonable
- experts disagree about the correct interpretation

For ambiguous examples, possible actions include:

- improving the labeling guideline
- using multiple annotators
- introducing an `uncertain` category
- removing the example from evaluation
- asking an expert to make the final decision

The distinction is:

- **Incorrect label:** a reasonably clear correct label exists, but the stored label is wrong.
- **Ambiguous example:** the correct label is uncertain even for qualified annotators.

## 8. Practical Workflow

A practical process is:

1. Add an `incorrectly labeled` category during dev-set error analysis.
2. Estimate how many observed errors are caused by incorrect labels.
3. Determine whether they materially affect model comparison or project decisions.
4. Decide whether to clean the training set, evaluation sets, or a specific problematic data source.
5. Apply the same labeling rules consistently to both dev and test sets.
6. If performing formal cleaning, review examples independently of the current model's predictions.

## 9. Common Mistakes

### Cleaning the Entire Training Set Immediately

This may require substantial effort even when random training-label noise has little effect on performance.

### Correcting Only Dev-Set Errors

This can make the evaluation set dependent on the current model and overlook incorrect labels among apparently correct predictions.

### Cleaning Dev but Not Test

This creates inconsistent annotation quality between the two evaluation sets.

### Treating Every Disagreement as a Labeling Error

Some examples are genuinely ambiguous and require clearer guidelines rather than a simple label correction.

### Ignoring Systematic Noise

Even a relatively small amount of label noise may be harmful if it is concentrated in an important class or data source.

## 10. Essence

The essence of cleaning incorrectly labeled data is:

> Do not clean labels automatically simply because some errors exist. Measure whether incorrect labels significantly affect training or evaluation, and clean them only when the expected benefit justifies the effort.

The three main principles are:

1. Deep neural networks can often tolerate a small amount of random label noise in the training set.
2. Incorrect dev/test labels are especially important because they can lead to incorrect model-selection decisions.
3. If dev/test labels are cleaned, the process must be consistent and should not examine only examples that the current model predicts incorrectly.

# Build Your First System Quickly, Then Iterate

## 1. Core Idea

The main principle is:

> Build a reasonable first system quickly, evaluate it, analyze its errors, and use the evidence to decide what to improve next.

At the beginning of a machine learning project, the team usually does not yet know:

- which failure modes are most common
- whether the main problem is bias, variance, or data mismatch
- whether the bottleneck lies in the model, data, or pipeline
- which proposed improvement has the greatest potential impact

A first working system provides the evidence needed to answer these questions.

## 2. Why Build the First System Quickly?

Before a baseline exists, proposed improvements are mostly based on intuition.

For example, a team may consider:

- building a specialized dog detector
- collecting more blurry images
- increasing model size
- cleaning all training labels
- designing a more complex architecture

After building a baseline and examining its errors, the team may discover that some of these problems account for only a small fraction of the total error.

Therefore:

> A baseline replaces speculation with empirical evidence.

## 3. What “Quickly” Means

Building quickly does not mean building carelessly.

The first system should be simple enough to implement and debug, but reliable enough to produce meaningful results.

A reasonable first system should have:

- a complete input-to-output pipeline
- correct training and evaluation code
- an appropriate metric
- a representative dev set
- no obvious data leakage
- predictions that can be inspected manually

If the implementation is fundamentally incorrect, its errors will not provide useful information.

## 4. The Iterative Workflow

The recommended workflow is:

$$
\text{Build}
\rightarrow
\text{Evaluate}
\rightarrow
\text{Diagnose}
\rightarrow
\text{Analyze Errors}
\rightarrow
\text{Improve}
\rightarrow
\text{Repeat}
$$

### Step 1: Define the Goal

Choose:

- the evaluation metric
- the dev and test distributions
- optimizing and satisficing metrics
- the production requirements

### Step 2: Build a Reasonable Baseline

Use a simple and established approach that can produce predictions quickly.

The goal is not to achieve state-of-the-art performance immediately. The goal is to create a useful reference point.

### Step 3: Evaluate the System

Measure performance on the training and dev sets.

These results help diagnose broad problems such as:

- avoidable bias
- variance
- data mismatch

### Step 4: Perform Error Analysis

Inspect a representative sample of dev-set errors.

Identify:

- common error categories
- incorrectly labeled examples
- important failure modes
- categories with large potential improvement

### Step 5: Choose the Next Improvement

Select an intervention based on the diagnosis.

Examples:

- high avoidable bias → increase model capacity or improve optimization
- high variance → add data or regularization
- data mismatch → collect more target-distribution data
- common blurry-input errors → use suitable augmentation or collect blurry examples
- frequent retrieval failures → improve the retriever

### Step 6: Evaluate Again

After the change:

1. compare the new system with the previous baseline
2. determine whether the hypothesis was supported
3. examine the new error distribution
4. choose the next improvement

The improved system becomes the new baseline.

## 5. Relationship to Error Analysis

Building the first system and carrying out error analysis are closely connected.

The first system produces:

- predictions
- metrics
- real failure cases

Error analysis converts those failures into engineering priorities.

Therefore:

> A baseline creates evidence, and error analysis converts that evidence into decisions.

Without a baseline, there are no real errors to inspect.

Without error analysis, the team may continue improving the wrong part of the system.

## 6. Keep Experiments Interpretable

During iteration, avoid changing too many components at the same time.

If the team simultaneously changes:

- architecture
- preprocessing
- training data
- loss function
- learning rate

and performance improves, it becomes difficult to determine which change caused the improvement.

Whenever practical, each experiment should test a clear hypothesis with a limited number of changes.

For example:

> Hypothesis: motion blur is a major source of error.  
> Intervention: add realistic motion-blur augmentation.  
> Evaluation: compare overall dev error and blurry-image error before and after the change.

## 7. When More Upfront Planning Is Necessary

Building quickly does not mean immediately deploying an immature system.

More initial planning is necessary when:

- mistakes can create serious safety risks
- data collection is extremely expensive
- hardware or infrastructure changes take a long time
- privacy or legal requirements must be satisfied
- the team already has strong experience with a similar problem

Even in these cases, it is still valuable to establish an evaluation pipeline and obtain empirical feedback as early as possible.

## 8. Common Mistakes

### Building an Overly Complex First System

Adding many components before knowing whether they address important failure modes increases development time and makes debugging harder.

### Using an Unreliable Baseline

A baseline with incorrect preprocessing, data leakage, or broken evaluation produces misleading evidence.

### Tuning Without Inspecting Errors

Repeatedly adjusting hyperparameters based only on an overall metric may provide little insight into the real bottleneck.

### Changing Too Many Things at Once

This makes experimental results difficult to interpret.

### Focusing Only on the Model

The main problem may instead lie in:

- data quality
- labeling
- preprocessing
- evaluation
- the dev-set distribution
- another component in the pipeline

## 9. Essence

The essence of **Build Your First System Quickly, Then Iterate** is:

> Do not attempt to design the perfect system based only on assumptions. Build a reasonable baseline, learn from its actual errors, make an evidence-based improvement, and repeat.

The central workflow is:

$$
\boxed{
\text{Build}
\rightarrow
\text{Measure}
\rightarrow
\text{Diagnose}
\rightarrow
\text{Improve}
\rightarrow
\text{Repeat}
}
$$

The main lesson is:

> The speed at which a team learns from experiments is often more valuable than the sophistication of its initial design.

# Bias and Variance with Mismatched Data Distributions

## 1. Core Idea

When the training set and dev set come from different distributions, the gap between training error and dev error cannot be interpreted as variance alone.

It may reflect:

- high variance
- data mismatch
- both problems

To separate these effects, we introduce a **train-dev set**.

## 2. The Train-Dev Set

A train-dev set:

- comes from the same distribution as the training set
- is not used to update model parameters
- is used only for evaluation

A typical setup is:

| Dataset | Distribution | Purpose |
|---|---|---|
| Train | Source/training distribution | Learn parameters |
| Train-dev | Source/training distribution | Diagnose variance |
| Dev | Target distribution | Select models |
| Test | Target distribution | Final evaluation |

For example:

- Train: internet images
- Train-dev: unseen internet images
- Dev: mobile-user images
- Test: mobile-user images

## 3. Diagnostic Workflow

Compare the errors in this order:

$$
E_{\text{human}}
\rightarrow
E_{\text{train}}
\rightarrow
E_{\text{train-dev}}
\rightarrow
E_{\text{dev}}
$$

Each gap represents a different diagnostic question.

## 4. Avoidable Bias

Compare human-level error with training error:

$$
\text{Avoidable bias}
\approx
E_{\text{train}}-E_{\text{human}}
$$

A large gap means that the model does not fit the training set as well as it reasonably could.

Possible actions include:

- use a larger model
- improve the architecture
- train longer
- improve optimization
- reduce excessive regularization

## 5. Variance

Compare training error with train-dev error:

$$
\text{Variance gap}
\approx
E_{\text{train-dev}}-E_{\text{train}}
$$

Because train and train-dev come from the same distribution, a large gap indicates poor generalization within the training distribution.

Possible actions include:

- collect more training data
- use regularization
- apply data augmentation
- reduce model complexity when appropriate

## 6. Data Mismatch

Compare train-dev error with dev error:

$$
\text{Data mismatch gap}
\approx
E_{\text{dev}}-E_{\text{train-dev}}
$$

Both sets contain examples that were not used for training, but they come from different distributions.

A large gap suggests that the model performs well on the source distribution but poorly on the target distribution.

Possible actions include:

- collect more target-like data
- include more target data in training
- use augmentation that resembles the target distribution
- fine-tune on target-domain data

## 7. Important Diagnostic Patterns

### High Avoidable Bias

| Dataset | Error |
|---|---:|
| Human-level | 1% |
| Train | 15% |
| Train-dev | 16% |
| Dev | 17% |

The main gap is:

$$
15\%-1\%=14\%
$$

The primary problem is high avoidable bias.

### High Variance

| Dataset | Error |
|---|---:|
| Human-level | 1% |
| Train | 1% |
| Train-dev | 10% |
| Dev | 11% |

The main gap is:

$$
10\%-1\%=9\%
$$

The primary problem is high variance.

### Data Mismatch

| Dataset | Error |
|---|---:|
| Human-level | 1% |
| Train | 1% |
| Train-dev | 2% |
| Dev | 12% |

The main gap is:

$$
12\%-2\%=10\%
$$

The primary problem is data mismatch.

### Variance and Data Mismatch

| Dataset | Error |
|---|---:|
| Human-level | 1% |
| Train | 1% |
| Train-dev | 8% |
| Dev | 15% |

Variance gap:

$$
8\%-1\%=7\%
$$

Data mismatch gap:

$$
15\%-8\%=7\%
$$

The system has both high variance and data mismatch.

## 8. Error Analysis for Data Mismatch

A large train-dev-to-dev gap shows that the distributions differ in a way that affects performance, but it does not explain the specific cause.

To understand the mismatch, compare error categories on train-dev and dev.

For example:

| Error Category | Train-Dev | Dev |
|---|---:|---:|
| Blurry images | 5% | 35% |
| Low-light images | 3% | 25% |
| Occluded objects | 10% | 12% |

This suggests that blurry and low-light images are important contributors to the mismatch.

## 9. Important Cautions

### Do Not Diagnose Variance from Train and Dev Alone

When train and dev come from different distributions:

$$
E_{\text{dev}}-E_{\text{train}}
$$

contains both variance and data-mismatch effects.

### Train-Dev Must Be Unseen

Train-dev examples must not be used to update model parameters.

### Train and Train-Dev Must Match

If train-dev does not represent the training distribution, it cannot reliably measure variance.

### These Gaps Are Diagnostic Approximations

The interpretation assumes that:

- train and train-dev come from the same distribution
- dev represents the target distribution
- the same metric is used on all sets
- labeling quality is reasonably consistent

## 10. Essence

The essence of bias and variance diagnosis with mismatched distributions is:

> Introduce a train-dev set from the training distribution so that variance can be separated from data mismatch.

The three key comparisons are:

$$
E_{\text{train}}-E_{\text{human}}
\quad\Rightarrow\quad
\text{Avoidable bias}
$$

$$
E_{\text{train-dev}}-E_{\text{train}}
\quad\Rightarrow\quad
\text{Variance}
$$

$$
E_{\text{dev}}-E_{\text{train-dev}}
\quad\Rightarrow\quad
\text{Data mismatch}
$$

The workflow to remember is:

$$
\boxed{
\text{Human-level}
\rightarrow
\text{Train}
\rightarrow
\text{Train-dev}
\rightarrow
\text{Dev}
}
$$

# Addressing Data Mismatch

## 1. Core Idea

After introducing a train-dev set, a large gap between train-dev error and dev error suggests data mismatch:

$$
E_{\text{dev}} \gg E_{\text{train-dev}}
$$

This means the model performs well on the training distribution but poorly on the target distribution.

The main goal is:

> Make the training data more similar to the target distribution.

## 2. Identify the Specific Mismatch

Knowing that data mismatch exists is not enough.

The training and target distributions may differ in:

- image quality
- background noise
- recording device
- lighting conditions
- language style
- document format
- user behavior

To determine the important differences, perform error analysis on both:

- the train-dev set
- the dev set

Then compare which error categories occur more frequently on the dev set.

For example:

| Error Category | Train-Dev | Dev |
|---|---:|---:|
| Car noise | 2% | 35% |
| Street noise | 5% | 8% |
| Low-volume speech | 10% | 12% |

This suggests that car noise is an important source of mismatch.

## 3. Collect More Target-Like Data

The most direct solution is to collect more data from the target distribution.

Examples include:

- collect mobile images when production uses mobile images
- record speech inside cars when production contains car audio
- collect real user questions when training data is synthetic
- collect documents in the same format used in production

The updated training set may contain both source and target-like data:

$$
D_{\text{train}}
=
D_{\text{source}}
\cup
D_{\text{target-like}}
$$

Enough target data should still be reserved for the dev and test sets.

## 4. Artificial Data Synthesis

When real target data is difficult or expensive to collect, artificial data synthesis can be used to make source examples resemble target examples.

For speech recognition, clean speech may be combined with car noise:

$$
x_{\text{synthetic}}
=
x_{\text{clean speech}}
+
x_{\text{car noise}}
$$

For image tasks, possible transformations include:

- blur
- low-light effects
- camera noise
- compression artifacts
- realistic occlusion

The transformations should be based on observed mismatch rather than applied only because they are commonly used.

## 5. Limited Diversity in Synthetic Data

A major risk is generating a large synthetic dataset from only a small number of original sources.

For example, one hour of car noise may be combined with thousands of hours of clean speech.

Although this creates many synthetic examples, the independent information about car noise still comes from only one hour of recordings.

Therefore:

> A large amount of synthetic data does not necessarily provide high diversity.

Synthetic data should include:

- multiple noise sources
- different environments
- different devices
- different intensity levels
- varied transformations

Otherwise, the model may overfit to a narrow artificial distribution.

## 6. Practical Workflow

A practical process is:

1. Compare train-dev and dev performance.
2. Confirm that a meaningful mismatch exists.
3. Compare train-dev and dev error categories.
4. Identify the most important differences.
5. Collect more real target-like data when possible.
6. Create realistic synthetic data when necessary.
7. Retrain the model.
8. Evaluate both overall dev performance and the targeted error category.

The workflow is:

$$
\text{Detect Mismatch}
\rightarrow
\text{Analyze Differences}
\rightarrow
\text{Collect or Synthesize Target-Like Data}
\rightarrow
\text{Retrain}
\rightarrow
\text{Evaluate}
$$

## 7. Example: Legal QA

Suppose:

- training data mainly contains synthetic questions
- dev data contains real user questions

Error analysis may show that real questions contain:

- spelling mistakes
- informal language
- missing legal-document names
- incomplete information
- multiple conditions in one question

Possible solutions include:

- adding real user questions to the training set
- paraphrasing synthetic questions into conversational language
- introducing realistic spelling variation
- creating questions with incomplete but natural wording

Any synthetic transformation must preserve the original legal meaning and correct answer.

## 8. Common Mistakes

### Collecting More Data from the Wrong Distribution

Adding more clean source data does not necessarily reduce target-domain mismatch.

### Creating Synthetic Data Without Error Analysis

Synthetic transformations should reflect actual dev-set failure modes.

### Confusing Quantity with Diversity

Many examples generated from a few original sources may still represent a narrow distribution.

### Creating Unrealistic Transformations

The model may learn artificial patterns that do not occur in production.

### Evaluating Only Overall Performance

After addressing a mismatch, evaluate both:

- overall dev performance
- performance on the targeted error category

## 9. Modern Extensions

### Fine-Tuning on Target Data

A common strategy is:

1. train or pretrain on a large source dataset
2. fine-tune on a smaller target-domain dataset

This allows the model to retain broad representations while adapting to the target distribution.

### Reweighting Target-Like Examples

When target-like examples are rare, they may be overwhelmed by source data.

Possible approaches include:

- oversampling target examples
- assigning them larger training weights
- controlling the source-to-target ratio in mini-batches

Care is required because a small target dataset can be overfit easily.

## 10. Essence

The essence of addressing data mismatch is:

> Identify how the target distribution differs from the training distribution, and then make the training data more representative of the target domain.

The three main actions are:

1. use error analysis to identify the mismatch
2. collect more real target-like data
3. create realistic and diverse synthetic data when necessary

The key warning is:

> A large synthetic dataset is not necessarily diverse if it is generated from only a small number of original sources.

# Transfer Learning

## 1. Core Idea

**Transfer learning** is the process of reusing knowledge learned from one task to help another related task.

Suppose:

- Task A is the source task.
- Task B is the target task.
- Task A has a large amount of training data.
- Task B has less training data.

Instead of training a model for Task B from random initialization, we begin with parameters learned from Task A:

$$
\theta_B^{(0)}
\leftarrow
\theta_A^*
$$

The central idea is:

> Representations learned from a large source task may also be useful for a related target task.

## 2. Why Transfer Learning Works

The hidden layers of a neural network learn intermediate representations.

For example, a computer-vision model may learn:

- edges
- curves
- textures
- shapes
- object parts

These features may be useful for multiple image tasks, even when the output classes are different.

Therefore, the early and intermediate layers of a network can often be reused for a new task.

## 3. Basic Procedure

Assume a neural network has been trained on Task A:

$$
x
\rightarrow
h^{[1]}
\rightarrow
h^{[2]}
\rightarrow
\cdots
\rightarrow
h^{[L-1]}
\rightarrow
\hat{y}_A
$$

To transfer the model to Task B:

1. Take the network trained on Task A.
2. Remove the task-specific output layer.
3. Replace it with a new output layer for Task B.
4. Initialize the new output layer randomly.
5. Train the model using Task B data.

The transferred model becomes:

$$
x
\rightarrow
h^{[1]}
\rightarrow
h^{[2]}
\rightarrow
\cdots
\rightarrow
h^{[L-1]}
\rightarrow
\hat{y}_B
$$

## 4. Training with a Small Target Dataset

If Task B has very little data, it is often useful to freeze the transferred hidden layers and train only the new output layer.

The pretrained network acts as a fixed feature extractor:

$$
h
=
f_{\text{pretrained}}(x)
$$

The target task learns only a new prediction function:

$$
\hat{y}_B
=
g(W_Bh+b_B)
$$

This reduces the number of trainable parameters and helps prevent overfitting.

## 5. Training with More Target Data

If Task B has more data, it may be useful to update some or all of the transferred layers.

Possible strategies include:

- train only the new output layer
- train the output layer and the last few hidden layers
- train the entire network

The more target data is available, the more reasonable it becomes to update a larger part of the network.

## 6. When Transfer Learning Is Useful

Transfer learning is most likely to help when the following conditions hold.

### The Tasks Have the Same Type of Input

Examples include:

- image task to image task
- speech task to speech task
- text task to text task

### The Source Task Has Much More Data

Typically:

$$
m_A \gg m_B
$$

The large source dataset allows the model to learn useful representations that the smaller target dataset could not learn reliably from scratch.

### Low-Level Features Are Useful for Both Tasks

For example:

- visual edges and textures may help multiple image tasks
- phonetic features may help multiple speech tasks
- language representations may help multiple text tasks

The tasks do not need to have the same output labels, but they should share useful underlying representations.

## 7. Example

Suppose Task A is image classification with one million labeled images and 1,000 output classes.

Task B is medical-image classification with only a few thousand labeled examples and two output classes.

A practical approach is:

1. use the network trained on Task A
2. remove its 1,000-class output layer
3. add a new two-class output layer
4. train the new layer on Task B
5. optionally update additional layers if enough target data is available

The pretrained model provides useful visual features, while the new output layer learns the target classification task.

## 8. Transfer Learning vs. Training from Scratch

Training from scratch begins with random parameters:

$$
\theta^{(0)}
=
\theta_{\text{random}}
$$

Transfer learning begins with previously learned parameters:

$$
\theta^{(0)}
=
\theta_{\text{pretrained}}
$$

Starting from pretrained parameters can provide:

- faster convergence
- better performance with limited target data
- reduced variance
- more useful initial representations

## 9. Relationship to Multi-Task Learning

Transfer learning trains tasks sequentially:

$$
\text{Task A}
\rightarrow
\text{Task B}
$$

Knowledge is first learned from Task A and then reused for Task B.

Multi-task learning trains multiple tasks at the same time using shared representations.

Therefore, transfer learning is sequential, while multi-task learning is simultaneous.

## 10. Essence

The essence of transfer learning is:

> Learn useful representations from a data-rich source task and reuse them to improve learning on a related target task.

The basic workflow is:

$$
\boxed{
\text{Train on Task A}
\rightarrow
\text{Replace the Output Layer}
\rightarrow
\text{Train on Task B}
\rightarrow
\text{Update More Layers if Needed}
}
$$

Transfer learning is especially useful when:

1. Task A and Task B have the same type of input.
2. Task A has much more data than Task B.
3. The representations learned from Task A are useful for Task B.

# Transfer Learning and Fine-Tuning

## 1. Core Distinction

**Transfer learning** is the broader strategy of reusing knowledge learned from a source task or dataset to improve performance on a related target task.

**Fine-tuning** is a specific adaptation method in which a pretrained model is further optimized on target-task data.

The relationship is:

$$
\boxed{
\text{Fine-tuning is usually a form of transfer learning}
}
$$

However:

$$
\boxed{
\text{Not all transfer learning requires fine-tuning the pretrained backbone}
}
$$

A concise distinction is:

> Transfer learning describes the reuse of previously learned knowledge, while fine-tuning describes how a pretrained model is further optimized for the target task.

## 2. Transfer Learning

Suppose a model is trained on a source task, Task A:

$$
\theta_A^*
=
\arg\min_\theta J_A(\theta)
$$

The learned parameters or representations are then reused for a target task, Task B.

The transferred knowledge may include:

- pretrained weights
- learned embeddings
- an encoder or backbone
- low-level features
- intermediate representations

The general process is:

$$
\text{Knowledge Learned from Task A}
\rightarrow
\text{Task B}
$$

Transfer learning does not specify exactly which parameters must be updated on Task B.

## 3. Fine-Tuning

Fine-tuning begins with pretrained parameters rather than random initialization:

$$
\theta^{(0)}
=
\theta_{\text{pretrained}}
$$

The model is then optimized on the target objective:

$$
\theta^*
=
\arg\min_\theta J_{\text{target}}(\theta)
$$

Fine-tuning may update:

- all pretrained parameters
- only selected pretrained layers
- only additional adapter parameters

The main purpose is to adapt general pretrained representations to the target task or domain.

## 4. Frozen Feature Extraction

A pretrained model can be used as a fixed feature extractor.

The backbone remains frozen:

$$
\theta_{\text{backbone}}
=
\text{constant}
$$

It produces a representation:

$$
h
=
f_{\theta_{\text{backbone}}}(x)
$$

Only a new task-specific head is trained:

$$
\hat{y}
=
g_\phi(h)
$$

In this case:

- transfer learning: **yes**
- fine-tuning the backbone: **no**
- feature extraction: **yes**

Knowledge is transferred through the frozen representations even though the pretrained backbone is not updated.

## 5. Partial Fine-Tuning

In partial fine-tuning, some pretrained layers remain frozen while selected layers are updated.

For example:

$$
\theta_{\text{early}}
=
\text{frozen}
$$

$$
\theta_{\text{late}}
=
\text{trainable}
$$

This is useful when:

- the target dataset has a moderate size
- early-layer features remain useful
- later layers require adaptation
- full fine-tuning is too expensive
- full fine-tuning may overfit

Partial fine-tuning is both transfer learning and fine-tuning.

## 6. Full Fine-Tuning

In full fine-tuning, all pretrained parameters are updated:

$$
\theta
\leftarrow
\theta
-
\alpha\nabla_\theta J_{\text{target}}
$$

Advantages include:

- maximum adaptation capacity
- all representations can change
- stronger adaptation to a different target domain

Disadvantages include:

- high GPU-memory requirements
- large optimizer states
- greater computational cost
- larger task-specific checkpoints
- greater risk of overfitting
- possible degradation of previously learned capabilities

## 7. Parameter-Efficient Fine-Tuning

**Parameter-efficient fine-tuning**, or PEFT, adapts a pretrained model while updating only a small number of parameters.

Most base-model parameters remain frozen.

Common PEFT methods include:

- LoRA
- QLoRA
- adapters
- prompt tuning
- prefix tuning

PEFT can reduce:

- memory usage
- training cost
- checkpoint size
- storage requirements for multiple tasks

PEFT is still fine-tuning because trainable parameters are optimized on target data.


## 8. LoRA

LoRA keeps the original pretrained weight matrix frozen:

$$
W
\in
\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}
$$

Instead of updating \(W\) directly, it learns a low-rank update:

$$
W'
=
W+\Delta W
$$

where:

$$
\Delta W
=
BA
$$

with:

$$
A
\in
\mathbb{R}^{r\times d_{\text{in}}}
$$

$$
B
\in
\mathbb{R}^{d_{\text{out}}\times r}
$$

and:

$$
r
\ll
\min(d_{\text{in}},d_{\text{out}})
$$

Only the smaller matrices \(A\) and \(B\) are trained.

LoRA is simultaneously:

- transfer learning
- fine-tuning
- parameter-efficient fine-tuning

## 9. QLoRA

QLoRA combines:

- a quantized frozen base model
- trainable LoRA adapters

The structure is:

$$
\text{Quantized Frozen Base Model}
+
\text{Trainable LoRA Adapters}
$$

Quantization reduces the memory required to store and operate the base model.

LoRA remains the mechanism used to adapt the model.

QLoRA is especially useful when GPU memory is limited.

## 10. Comparison

| Situation | Transfer Learning | Fine-Tuning |
|---|---:|---:|
| Use a frozen pretrained encoder and train a new classifier | Yes | No backbone fine-tuning |
| Update only the last few pretrained layers | Yes | Yes, partial fine-tuning |
| Update all pretrained parameters | Yes | Yes, full fine-tuning |
| Train LoRA adapters while freezing the base model | Yes | Yes, PEFT |
| Train a model from random initialization | No | No pretrained fine-tuning |

Transfer learning is therefore the broader concept.

Fine-tuning is one family of adaptation methods within transfer learning.

## 11. Why LLM Work Usually Uses the Term Fine-Tuning

Large language models are already pretrained on large text corpora.

When a model such as Llama, Qwen, Gemma, or Mistral is adapted to a target task, the transfer-learning setting is already implied.

The practical discussion therefore focuses on the adaptation step:

> Fine-tune the pretrained language model.

Conceptually, the process is still:

$$
\text{General Language Knowledge}
\rightarrow
\text{Target Domain, Task, or Behavior}
$$


## 12. Two Independent Dimensions of LLM Fine-Tuning

LLM adaptation methods should be classified along two independent dimensions.

### Dimension 1: Training Objective

This describes what the model is learning.

Examples include:

- continued pretraining
- supervised fine-tuning
- preference optimization

### Dimension 2: Parameter-Update Strategy

This describes which parameters are updated.

Examples include:

- full fine-tuning
- partial fine-tuning
- LoRA
- QLoRA
- prompt tuning
- prefix tuning

These dimensions can be combined.

Examples include:

- supervised fine-tuning with full parameter updates
- supervised fine-tuning with LoRA
- DPO with LoRA
- continued pretraining with full parameter updates

Therefore:

> SFT, DPO, and continued pretraining describe training objectives, while LoRA and full fine-tuning describe parameter-update strategies.

## 13. Continued Pretraining

Continued pretraining uses a language-modeling objective on a new corpus.

For an autoregressive language model:

$$
\mathcal{L}_{\text{LM}}
=
-\sum_t
\log P_\theta(x_t\mid x_{<t})
$$

Example:

$$
\text{General Language Model}
\rightarrow
\text{Training on Legal Documents}
$$

Its purpose is to adapt the model to:

- domain vocabulary
- domain-specific writing style
- document structure
- domain knowledge distribution

Continued pretraining does not necessarily teach the model how to follow instructions.

## 14. Supervised Fine-Tuning

Supervised fine-tuning, or SFT, uses input-output or conversational examples.

The objective is typically:

$$
\mathcal{L}_{\text{SFT}}
=
-\sum_t
\log P_\theta
\left(
y_t
\mid
x,y_{<t}
\right)
$$

SFT can teach the model to:

- follow instructions
- perform a specific task
- produce a required response format
- generate structured output
- follow a domain-specific workflow

SFT describes the objective and training-data format.

It does not specify whether the model uses full fine-tuning, LoRA, or another parameter-update method.

## 15. Preference Optimization

Preference optimization uses comparisons between preferred and rejected responses.

A typical training example contains:

```plaintext
    Prompt
    ├── Preferred response
    └── Rejected response
```

Common approaches include:

- DPO
- RLHF
- ORPO
- KTO

The purpose is to make the model prefer responses that better satisfy desired quality criteria.

Preference optimization is usually considered part of LLM post-training.

It can be combined with either full fine-tuning or PEFT.

## 16. Conceptual Hierarchy

The relationship between transfer learning and parameter-update strategies can be represented as:

```plaintext
    Transfer Learning
    │
    ├── Frozen Feature Extraction
    │   └── Pretrained backbone remains frozen
    │
    └── Fine-Tuning
        │
        ├── Partial Fine-Tuning
        ├── Full Fine-Tuning
        └── Parameter-Efficient Fine-Tuning
            ├── LoRA
            ├── QLoRA
            ├── Adapters
            ├── Prompt Tuning
            └── Prefix Tuning
```

LLM adaptation can also be organized by training objective:

```plaintext
    LLM Adaptation and Post-Training
    │
    ├── Continued Pretraining
    ├── Supervised Fine-Tuning
    └── Preference Optimization
        ├── DPO
        ├── RLHF
        └── Other Preference Objectives
```

The first hierarchy describes which parameters are updated.

The second hierarchy describes what objective is optimized.

## 17. Practical Examples

### Frozen BERT Encoder

BERT is frozen and only a new classifier is trained.

- Transfer learning: Yes
- Fine-tuning BERT: No
- Feature extraction: Yes

### Full BERT Fine-Tuning

All BERT parameters are updated on a sentiment-classification dataset.

- Transfer learning: Yes
- Fine-tuning: Yes
- Full fine-tuning: Yes

### LoRA on Qwen

The Qwen base model is frozen and LoRA adapters are trained on Legal QA data.

- Transfer learning: Yes
- Fine-tuning: Yes
- Full fine-tuning: No
- PEFT: Yes

### Continued Pretraining Followed by SFT

A language model is:

1. continued-pretrained on legal documents
2. supervised-fine-tuned on legal question-answer pairs

The first stage adapts the model to legal language and domain knowledge.

The second stage teaches task-specific and instruction-following behavior.

## 18. Common Frameworks and Toolkits

### PyTorch

PyTorch is the underlying deep-learning framework for:

- custom training loops
- automatic differentiation
- optimizers
- mixed-precision training
- distributed training
- custom model architectures

It is suitable when detailed control over the training process is required.

### Hugging Face Transformers

Transformers provides:

- pretrained model architectures
- tokenizers
- task-specific model classes
- model loading
- the `Trainer` API
- model-hub integration

It is a common foundation for Transformer fine-tuning.

### Hugging Face PEFT

PEFT supports parameter-efficient methods such as:

- LoRA
- AdaLoRA
- IA3
- prompt tuning
- prefix tuning

It is useful when memory, storage, or compute is limited.

### Hugging Face TRL

TRL focuses on LLM post-training workflows such as:

- supervised fine-tuning
- reward modeling
- DPO
- preference optimization

### Hugging Face Accelerate

Accelerate provides infrastructure for:

- single-GPU training
- multi-GPU training
- multi-node training
- FSDP
- DeepSpeed

It is not a fine-tuning method itself.

### Axolotl

Axolotl provides configuration-driven LLM training and fine-tuning.

It is useful for:

- YAML-based experiments
- LoRA
- full fine-tuning
- distributed training
- reproducible pipelines

### LLaMA Factory

LLaMA Factory provides high-level tools for fine-tuning LLMs and VLMs.

It includes:

- configuration files
- command-line tools
- a web interface
- support for multiple adaptation methods

### Unsloth

Unsloth focuses on improving fine-tuning speed and memory efficiency.

It is commonly used for:

- LoRA
- QLoRA
- full fine-tuning
- limited-GPU environments

## 19. Framework Selection

| Requirement | Suitable Tool |
|---|---|
| Learn or control the training loop deeply | PyTorch |
| Fine-tune standard Transformer models | Transformers |
| Use LoRA, adapters, or prompt tuning | PEFT |
| Perform SFT or preference optimization | TRL |
| Run distributed or multi-GPU training | Accelerate |
| Use configuration-driven pipelines | Axolotl |
| Use CLI or web-based fine-tuning | LLaMA Factory |
| Optimize training with limited GPU memory | Unsloth |

These tools are often combined.

A common stack is:

    PyTorch
    └── Transformers
        ├── PEFT
        ├── TRL
        └── Accelerate

## 20. Precise Reporting

Instead of writing:

> We used transfer learning.

A more precise description is:

> We initialized the model from pretrained weights and fine-tuned all parameters on the target dataset.

For frozen feature extraction:

> We used the pretrained encoder as a frozen feature extractor and trained only a new classification head.

For LoRA:

> We performed supervised fine-tuning using LoRA while keeping the base-model parameters frozen.

A complete description should answer:

1. Which pretrained model was used?
2. Which training objective was optimized?
3. Which parameters were updated?

## 21. Essence

The central relationship is:

$$
\boxed{
\text{Transfer learning is the broader strategy}
}
$$

$$
\boxed{
\text{Fine-tuning is an adaptation method within that strategy}
}
$$

Transfer learning describes the reuse of knowledge from a pretrained model.

Fine-tuning describes how that pretrained model is adapted through optimization on target data.

For LLMs, always distinguish between:

1. the training objective:
   - continued pretraining
   - supervised fine-tuning
   - preference optimization

2. the parameter-update strategy:
   - full fine-tuning
   - partial fine-tuning
   - LoRA or QLoRA
   - prompt tuning or prefix tuning

The shortest correct statement is:

> Transfer learning describes the transfer of knowledge; fine-tuning describes the optimization used to adapt that knowledge.

# Multi-Task Learning

## 1. Core Idea

**Multi-task learning** trains one model on several related tasks at the same time.

Instead of building a separate model for each task, the model uses:

- shared layers that learn common representations
- task-specific output heads

The basic architecture is:

$$
\text{Input}
\rightarrow
\text{Shared Representation}
\rightarrow
\begin{cases}
\text{Task 1 Head}\\
\text{Task 2 Head}\\
\vdots\\
\text{Task T Head}
\end{cases}
$$

The central idea is:

> Related tasks can help each other by sharing useful features.

## 2. Example

Suppose an image-processing system must detect:

- pedestrians
- cars
- stop signs
- traffic lights

The target for one image may be:

$$
y=
\begin{bmatrix}
1\\
1\\
0\\
1
\end{bmatrix}
$$

This means that the image contains:

- a pedestrian
- a car
- no stop sign
- a traffic light

The model predicts:

$$
\hat y=
\begin{bmatrix}
\hat y_1\\
\hat y_2\\
\hat y_3\\
\hat y_4
\end{bmatrix}
$$

Each output can represent:

$$
\hat y_j=P(y_j=1\mid x)
$$

## 3. Shared Representation and Task-Specific Heads

The shared network computes:

$$
h=f_{\theta_{\text{shared}}}(x)
$$

Each task has its own prediction head:

$$
\hat y_j=g_{\theta_j}(h)
$$

The complete model is:

$$
\hat y_j
=
g_{\theta_j}
\left(
f_{\theta_{\text{shared}}}(x)
\right)
$$

The shared parameters receive gradients from all tasks:

$$
\nabla_{\theta_{\text{shared}}}J
=
\sum_{j=1}^{T}
\lambda_j
\nabla_{\theta_{\text{shared}}}J_j
$$

Therefore, every task can influence the shared representation.

## 4. Multi-Task Loss

For multiple tasks, the total objective is usually a weighted sum:

$$
J
=
\sum_{j=1}^{T}
\lambda_jJ_j
$$

where:

- $J_j$ is the loss for task $j$
- $\lambda_j$ controls the importance of task $j$

For multiple binary outputs, the loss for one example may be:

$$
\mathcal L^{(i)}
=
-\sum_{j=1}^{T}
\left[
y_j^{(i)}\log \hat y_j^{(i)}
+
\left(1-y_j^{(i)}\right)
\log
\left(1-\hat y_j^{(i)}\right)
\right]
$$

If all tasks are equally important, the task weights may initially be set to:

$$
\lambda_j=1
$$

## 5. Why Multi-Task Learning Can Help

### Shared Features

Related tasks may require similar features.

For traffic-scene understanding, several tasks may benefit from:

- edges
- shapes
- object boundaries
- road context
- spatial relationships

A shared network learns these common features once instead of learning them independently for every task.

### Additional Supervision

One task may provide useful learning signals for another task.

For example, learning to detect cars may improve the shared representation of road scenes and indirectly help pedestrian detection.

### Regularization

A model trained on one small task may overfit task-specific patterns.

Training on multiple related tasks encourages the shared representation to capture more general patterns.

Therefore, multi-task learning can act as a form of regularization.

## 6. When Multi-Task Learning Is Useful

Multi-task learning is most likely to help when the following conditions hold.

### The Tasks Share Useful Features

The tasks should benefit from similar underlying representations.

Using the same input type is not enough. The tasks must share meaningful structure.

### The Tasks Have Sufficient Data

Each task should contribute enough training signal.

If one task has far more data than the others, it may dominate optimization.

### The Model Has Enough Capacity

The model must have sufficient capacity to learn:

- shared representations
- task-specific differences

If the network is too small, the tasks may compete for capacity and reduce one another's performance.

## 7. Missing Labels

Not every training example must have labels for every task.

For example:

$$
y^{(i)}
=
\begin{bmatrix}
1\\
?\\
0\\
1
\end{bmatrix}
$$

The missing label must not be treated as a negative label.

Define a mask:

$$
m_j^{(i)}
=
\begin{cases}
1, & \text{if the label is available}\\
0, & \text{if the label is missing}
\end{cases}
$$

The loss becomes:

$$
\mathcal L^{(i)}
=
-\sum_{j=1}^{T}
m_j^{(i)}
\left[
y_j^{(i)}\log \hat y_j^{(i)}
+
\left(1-y_j^{(i)}\right)
\log
\left(1-\hat y_j^{(i)}\right)
\right]
$$

Only available labels contribute to the loss.

## 8. Multi-Task Learning vs. Multi-Label Classification

### Multi-Label Classification

A single example can belong to several labels at the same time.

For example, one image may contain:

- a car
- a pedestrian
- a traffic light

### Multi-Task Learning

The model solves several tasks, which may have:

- different output types
- different losses
- different datasets
- different prediction heads

For example, one vision model may simultaneously perform:

- image classification
- object detection
- depth estimation
- semantic segmentation

Multi-label classification can be viewed as a simple form of multi-task learning, but multi-task learning is broader.

## 9. Multi-Task Learning vs. Transfer Learning

Transfer learning and multi-task learning both reuse knowledge, but their training procedures differ.

### Transfer Learning

Tasks are learned sequentially:

$$
\text{Task A}
\rightarrow
\text{Task B}
$$

The model first learns Task A and then transfers its knowledge to Task B.

### Multi-Task Learning

Tasks are learned simultaneously:

$$
\text{Task A}
+
\text{Task B}
+
\text{Task C}
$$

The tasks jointly update a shared representation.

| Aspect | Transfer Learning | Multi-Task Learning |
|---|---|---|
| Training order | Sequential | Simultaneous |
| Knowledge flow | Source task to target task | Tasks support one another |
| Architecture | Pretrain, then adapt | Shared backbone with multiple heads |
| Best setting | Source task has much more data | Several related tasks have useful data |

A concise distinction is:

$$
\boxed{
\text{Transfer learning: learn first, transfer later}
}
$$

$$
\boxed{
\text{Multi-task learning: learn together}
}
$$

## 10. Negative Transfer

Multi-task learning does not always improve every task.

**Negative transfer** occurs when learning one task reduces performance on another.

This can happen when:

- the tasks are weakly related
- the tasks require conflicting representations
- one task dominates the gradients
- loss scales are very different
- the model lacks sufficient capacity

Conflicting task gradients may satisfy:

$$
\nabla_\theta J_1
\cdot
\nabla_\theta J_2
<
0
$$

This means that an update that helps one task may harm another.

## 11. Task Weighting

The total objective is:

$$
J
=
\lambda_1J_1
+
\lambda_2J_2
+
\cdots
+
\lambda_TJ_T
$$

If one loss is much larger than the others, that task may dominate training.

The weights $\lambda_j$ may need to account for:

- task importance
- loss scale
- dataset size
- task difficulty
- gradient magnitude

Task weighting should be evaluated experimentally.

## 12. Practical Workflow

A practical multi-task learning process is:

1. Identify tasks that may share useful features.
2. Design a shared backbone.
3. Add task-specific output heads.
4. Define a loss for each task.
5. Combine the losses using suitable weights.
6. Mask missing labels.
7. Train the tasks jointly.
8. Compare each task against its single-task baseline.

The model should not be considered successful merely because its average score improves.

Each task must be evaluated separately.

## 13. Common Mistakes

### Combining Tasks Only Because They Use the Same Input Type

Two image tasks or two text tasks are not necessarily related enough to help each other.

### Skipping Single-Task Baselines

Without separate baselines, it is impossible to determine whether joint training actually helps.

### Ignoring Task Imbalance

A task with more examples or a larger loss may dominate the shared representation.

### Treating Missing Labels as Negative Labels

Missing means unknown, not absent.

### Using a Model with Insufficient Capacity

A small model may force tasks to compete for the same limited representation.

### Looking Only at Average Performance

Some tasks may improve while an important task becomes worse.

## 14. Modern Extensions

### Task-Specific Adapters

A model may combine:

- a shared backbone
- task-specific adapters
- task-specific output heads

This preserves common knowledge while allowing each task to learn specialized representations.

### Multi-Task Pretraining

A model may be pretrained using several objectives at once.

For example, a vision-language model may jointly learn:

- image-text matching
- contrastive learning
- caption generation

### Instruction Tuning

Instruction tuning often combines examples from many tasks, such as:

- classification
- summarization
- question answering
- extraction
- rewriting

These tasks are converted into a common instruction-response format and learned jointly.

This can be viewed as large-scale multi-task learning.

## 15. Essence

The essence of multi-task learning is:

> Train several related tasks jointly so that they share representations and provide useful learning signals to one another.

The basic architecture is:

$$
\boxed{
\text{Shared Backbone}
\rightarrow
\text{Multiple Task-Specific Heads}
}
$$

The total objective is:

$$
\boxed{
J
=
\sum_{j=1}^{T}
\lambda_jJ_j
}
$$

Multi-task learning is most likely to help when:

1. the tasks share useful features
2. the tasks provide sufficient training data
3. the model has enough capacity

The most important distinction is:

> Transfer learning learns tasks sequentially, while multi-task learning learns tasks simultaneously.

# Step-by-Step Multi-Task Learning Example with Three Tasks

## 1. Problem Setup

Suppose one neural network processes handwritten-digit images and solves three tasks simultaneously:

1. **Digit classification:** predict a digit from 0 to 9.
2. **Parity classification:** predict whether the digit is even or odd.
3. **Rotation classification:** predict whether the image is rotated by \(0^\circ\), \(90^\circ\), \(180^\circ\), or \(270^\circ\).

These tasks can share visual features such as edges, curves, and digit shapes.

The architecture is:

$$
x
\rightarrow
h=f_{\theta_s}(x)
\rightarrow
\begin{cases}
\hat y_1=g_{\theta_1}(h) & \text{Digit head}\\
\hat y_2=g_{\theta_2}(h) & \text{Parity head}\\
\hat y_3=g_{\theta_3}(h) & \text{Rotation head}
\end{cases}
$$

where:

- $(\theta_s)$ contains the shared-backbone parameters
- $(\theta_1,\theta_2,\theta_3)$ contain the task-specific head parameters

## 2. One Training Example

Suppose the input image contains the digit $(7)$, rotated by $(90^\circ)$.

The labels are:

$$
y_{\text{digit}}=7
$$

$$
y_{\text{parity}}=1
$$

where \(1\) represents an odd number, and:

$$
y_{\text{rotation}}=1
$$

where rotation class $(1)$ represents $(90^\circ)$.

The complete training example is:

$$
\left(
x,
y_{\text{digit}},
y_{\text{parity}},
y_{\text{rotation}}
\right)
=
(x,7,1,1)
$$

## 3. Step 1: Compute the Shared Representation

The image is passed through the shared backbone:

$$
h=f_{\theta_s}(x)
$$

For illustration, suppose the representation is:

$$
h=
\begin{bmatrix}
0.8\\
-0.3\\
1.2\\
0.5
\end{bmatrix}
$$

This representation is passed to all three task-specific heads.

## 4. Step 2: Compute the Three Predictions

### Digit Head

The digit head produces probabilities for ten classes:

$$
p_{\text{digit}}
=
\operatorname{softmax}(W_1h+b_1)
$$

Suppose the probability assigned to the correct digit is:

$$
P(y_{\text{digit}}=7\mid x)=0.4
$$

### Parity Head

The parity head produces a binary probability:

$$
p_{\text{parity}}
=
\sigma(W_2h+b_2)
$$

Suppose:

$$
P(y_{\text{parity}}=1\mid x)=0.8
$$

### Rotation Head

The rotation head produces probabilities for four classes:

$$
p_{\text{rotation}}
=
\operatorname{softmax}(W_3h+b_3)
$$

Suppose the probability assigned to the correct rotation is:

$$
P(y_{\text{rotation}}=1\mid x)=0.5
$$

## 5. Step 3: Compute One Loss per Task

### Digit Loss

Using categorical cross-entropy:

$$
J_1
=
-\log(0.4)
\approx
0.916
$$

### Parity Loss

Using binary cross-entropy:

$$
J_2
=
-\log(0.8)
\approx
0.223
$$

### Rotation Loss

Using categorical cross-entropy:

$$
J_3
=
-\log(0.5)
\approx
0.693
$$

The individual losses are:

| Task | Loss |
|---|---:|
| Digit classification | 0.916 |
| Parity classification | 0.223 |
| Rotation classification | 0.693 |

Each task still has its own prediction and loss.

## 6. Step 4: Combine the Task Losses

Suppose the task weights are:

$$
\lambda_1=1,\qquad
\lambda_2=0.5,\qquad
\lambda_3=0.8
$$

The total loss is:

$$
J
=
\lambda_1J_1
+
\lambda_2J_2
+
\lambda_3J_3
$$

Substituting the values:

$$
J
=
1(0.916)
+
0.5(0.223)
+
0.8(0.693)
$$

$$
J
\approx
1.582
$$

This scalar loss is used for backpropagation.

## 7. Step 5: Backpropagate Through the Task-Specific Heads

Each task-specific head receives gradients only from its own loss.

For the digit head:

$$
\nabla_{\theta_1}J
=
\lambda_1\nabla_{\theta_1}J_1
$$

For the parity head:

$$
\nabla_{\theta_2}J
=
\lambda_2\nabla_{\theta_2}J_2
$$

For the rotation head:

$$
\nabla_{\theta_3}J
=
\lambda_3\nabla_{\theta_3}J_3
$$

Therefore:

- the digit loss updates the digit head
- the parity loss updates the parity head
- the rotation loss updates the rotation head

The tasks do not directly update one another's heads.

## 8. Step 6: Backpropagate Through the Shared Backbone

The shared backbone influences all three predictions.

Therefore, it receives the weighted sum of all three task gradients:

$$
\nabla_{\theta_s}J
=
\lambda_1\nabla_{\theta_s}J_1
+
\lambda_2\nabla_{\theta_s}J_2
+
\lambda_3\nabla_{\theta_s}J_3
$$

Suppose one shared parameter \(w\) receives:

$$
\frac{\partial J_1}{\partial w}=0.6
$$

$$
\frac{\partial J_2}{\partial w}=-0.2
$$

$$
\frac{\partial J_3}{\partial w}=0.4
$$

The total gradient is:

$$
\frac{\partial J}{\partial w}
=
1(0.6)
+
0.5(-0.2)
+
0.8(0.4)
$$

$$
\frac{\partial J}{\partial w}
=
0.82
$$

With learning rate:

$$
\alpha=0.01
$$

the update is:

$$
w
\leftarrow
w-0.01(0.82)
$$

$$
w
\leftarrow
w-0.0082
$$

The shared parameter therefore follows a weighted compromise between the three tasks.

## 9. Gradient Agreement and Conflict

In the example:

- Task 1 produces a positive gradient.
- Task 3 also produces a positive gradient.
- Task 2 produces a negative gradient.

Tasks 1 and 3 prefer a similar update direction, while Task 2 prefers the opposite direction.

The final update combines all three signals.

This illustrates why multi-task learning can produce:

- positive transfer when task gradients agree
- negative transfer when task gradients conflict

## 10. Step 7: Update All Parameters

The optimizer updates the shared backbone:

$$
\theta_s
\leftarrow
\theta_s
-
\alpha\nabla_{\theta_s}J
$$

It also updates each task-specific head:

$$
\theta_1
\leftarrow
\theta_1
-
\alpha\lambda_1\nabla_{\theta_1}J_1
$$

$$
\theta_2
\leftarrow
\theta_2
-
\alpha\lambda_2\nabla_{\theta_2}J_2
$$

$$
\theta_3
\leftarrow
\theta_3
-
\alpha\lambda_3\nabla_{\theta_3}J_3
$$

After the update:

- the backbone is modified using information from all three tasks
- each head is modified using only its own task loss

## 11. Mini-Batch Training

In practice, each task loss is averaged over a mini-batch of \(B\) examples:

$$
J_1
=
\frac{1}{B}
\sum_{i=1}^{B}
\mathcal L_1^{(i)}
$$

$$
J_2
=
\frac{1}{B}
\sum_{i=1}^{B}
\mathcal L_2^{(i)}
$$

$$
J_3
=
\frac{1}{B}
\sum_{i=1}^{B}
\mathcal L_3^{(i)}
$$

The total mini-batch loss is:

$$
J
=
\lambda_1J_1
+
\lambda_2J_2
+
\lambda_3J_3
$$

One backward pass computes:

- separate gradients for the three heads
- the combined gradient for the shared backbone

## 12. PyTorch-Style Training Step

```python
    optimizer.zero_grad()

    features = backbone(images)

    digit_logits = digit_head(features)
    parity_logits = parity_head(features)
    rotation_logits = rotation_head(features)

    digit_loss = digit_criterion(
        digit_logits,
        digit_labels,
    )

    parity_loss = parity_criterion(
        parity_logits,
        parity_labels.float(),
    )

    rotation_loss = rotation_criterion(
        rotation_logits,
        rotation_labels,
    )

    total_loss = (
        1.0 * digit_loss
        + 0.5 * parity_loss
        + 0.8 * rotation_loss
    )

    total_loss.backward()
    optimizer.step()

```

Although `backward()` is called only once, automatic differentiation preserves the dependency structure and computes the correct gradient for every parameter.

## 13. Monitor Every Task Separately

The total loss alone is not sufficient.

The training process should record:

$$
J_1,\qquad J_2,\qquad J_3
$$

It should also track a metric for each task:

| Task | Example Metric |
|---|---|
| Digit classification | Accuracy |
| Parity classification | Accuracy or F1 |
| Rotation classification | Accuracy |

For example:

| Epoch | Digit Accuracy | Parity Accuracy | Rotation Accuracy |
|---:|---:|---:|---:|
| 1 | 70% | 91% | 62% |
| 5 | 90% | 97% | 78% |
| 10 | 94% | 98% | 79% |

If one task stops improving, possible causes include:

- its loss weight is too small
- it has too little data
- another task dominates the shared gradient
- the task gradients conflict
- the model lacks task-specific capacity

## 14. Complete Training Flow

The complete process is:

$$
\boxed{
\begin{aligned}
x
&\rightarrow
h=f_{\theta_s}(x)\\
h
&\rightarrow
\hat y_1,\hat y_2,\hat y_3\\
\hat y_1,\hat y_2,\hat y_3
&\rightarrow
J_1,J_2,J_3\\
J
&=
\lambda_1J_1
+
\lambda_2J_2
+
\lambda_3J_3\\
\nabla_{\theta_s}J
&=
\lambda_1\nabla_{\theta_s}J_1
+
\lambda_2\nabla_{\theta_s}J_2
+
\lambda_3\nabla_{\theta_s}J_3\\
\theta
&\leftarrow
\theta-\alpha\nabla_\theta J
\end{aligned}
}
$$

## 15. Essence

The key idea is:

> Each task has its own prediction, loss, head, and evaluation metric. The task gradients are combined only when updating the shared representation.

Therefore:

- task-specific heads learn independently from their own losses
- the shared backbone learns from the weighted sum of all task gradients
- task weights control relative influence
- agreeing gradients can create positive transfer
- conflicting gradients can create negative transfer

# What Is End-to-End Deep Learning?

## 1. Core Idea

**End-to-end deep learning** trains a model to map raw input directly to the final desired output.

Instead of manually designing several intermediate stages:

$$
x
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
\cdots
\rightarrow
y
$$

the model learns:

$$
\hat y=f_\theta(x)
$$

The parameters are optimized using the final task loss:

$$
\theta^*
=
\arg\min_\theta
\frac{1}{m}
\sum_{i=1}^{m}
\mathcal L
\left(
f_\theta(x^{(i)}),
y^{(i)}
\right)
$$

The central principle is:

> Learn the complete input-to-output mapping directly from data.

## 2. Traditional Pipeline vs. End-to-End Learning

### Traditional Pipeline

A traditional system may contain several manually designed stages:

$$
\text{Raw Input}
\rightarrow
\text{Hand-Designed Features}
\rightarrow
\text{Intermediate Predictions}
\rightarrow
\text{Final Output}
$$

Each stage may be trained or designed separately.

### End-to-End Learning

An end-to-end system learns the complete mapping:

$$
\text{Raw Input}
\rightarrow
\text{Neural Network}
\rightarrow
\text{Final Output}
$$

The model still learns intermediate representations, but they are not necessarily defined or labeled by humans.

## 3. Speech Recognition Example

A traditional speech-recognition pipeline may be:

$$
\text{Audio}
\rightarrow
\text{Acoustic Features}
\rightarrow
\text{Phonemes}
\rightarrow
\text{Words}
\rightarrow
\text{Transcript}
$$

An end-to-end system learns:

$$
\text{Audio}
\rightarrow
\text{Transcript}
$$

The training data contains pairs such as:

$$
(\text{audio},\text{transcript})
$$

The model learns the useful intermediate representations automatically.

## 4. Face Recognition Example

A modular approach may first compute embeddings:

$$
e_1=f_\theta(x_1)
$$

$$
e_2=f_\theta(x_2)
$$

and then compare their distance:

$$
\hat y
=
\mathbf{1}
\left[
d(e_1,e_2)<\tau
\right]
$$

A fully end-to-end approach could instead learn:

$$
(x_1,x_2)
\rightarrow
P(\text{same person})
$$

However, the embedding-based approach may be more practical because it can reuse identity-classification data and allows efficient comparison between many faces.

## 5. Autonomous Driving Example

A modular driving pipeline may be:

$$
\text{Camera Image}
\rightarrow
\text{Object Detection}
\rightarrow
\text{Motion Estimation}
\rightarrow
\text{Path Planning}
\rightarrow
\text{Steering Command}
$$

An end-to-end system attempts to learn:

$$
\text{Camera Image}
\rightarrow
\text{Steering Angle}
$$

This reduces manual intermediate design, but it requires a very large and diverse dataset of driving examples.

## 6. Advantages

### Learning Representations from Data

The model can learn intermediate features that are better suited to the final objective than manually designed representations.

### Direct Optimization of the Final Objective

The entire system is optimized for the output that actually matters:

$$
J_{\text{final}}
$$

This avoids optimizing intermediate objectives that may not align perfectly with final performance.

### Less Intermediate Labeling

An end-to-end system may require only final input-output pairs instead of labels for every intermediate stage.

### Joint Optimization

All trainable components can adapt together to improve final performance.

## 7. Limitations

### Large Data Requirements

The model must learn both:

- intermediate representations
- the final prediction mapping

Therefore, end-to-end learning often requires many labeled input-output examples.

### Difficulty Using Intermediate Supervision

A modular system can use separate datasets for object detection, speech phonemes, landmarks, or other intermediate tasks.

A pure end-to-end system may fail to use these additional labels effectively.

### Harder Error Analysis

When the final prediction is wrong, it may be difficult to determine whether the problem lies in:

- perception
- representation learning
- reasoning
- planning
- control

### Difficulty Enforcing Rules

Safety rules, physical constraints, and deterministic procedures may be easier to implement in explicit modules.

## 8. When End-to-End Learning Is Appropriate

End-to-end learning is more likely to work well when:

1. A large number of labeled input-output pairs is available.
2. The input contains enough information to predict the output.
3. The final objective can be defined clearly.
4. Useful intermediate representations are difficult to design manually.
5. The model has enough capacity to learn the full mapping.

## 9. When a Modular System May Be Better

A modular or hybrid system may be preferable when:

- final labeled data is limited
- useful intermediate labels are available
- some modules are already reliable
- interpretability and debugging are important
- strict safety constraints must be enforced
- some operations are better handled by deterministic algorithms

The decision is not between a modern and an outdated approach. It is a trade-off between data-driven learning and explicit structure.

## 10. End-to-End Architecture vs. End-to-End Training

### End-to-End Architecture

The system maps input directly to final output:

$$
x\rightarrow\hat y
$$

without an explicit human-designed intermediate interface.

### End-to-End Training

The final loss updates all trainable components:

$$
\nabla_\theta J_{\text{final}}
$$

A system may contain multiple neural modules and still be trained end-to-end if gradients flow through the entire trainable pipeline.

## 11. Modern Perspective

Many current systems are hybrid rather than purely end-to-end.

They may combine:

- pretrained neural models
- task-specific modules
- search or planning
- explicit rules
- external tools
- verification components

Pretrained foundation models can reduce the amount of labeled data required because they already contain useful representations. However, they do not remove the need for careful evaluation, error analysis, and system design.

## 12. Common Misconceptions

### End-to-End Is Always Better

False. It is effective only when enough suitable data is available.

### End-to-End Models Have No Intermediate Representations

False. They still learn internal representations, but humans do not explicitly define them.

### A Deep Network Is Automatically End-to-End

False. Depth alone does not determine whether a system is end-to-end.

### Modular Systems Are Obsolete

False. Modular systems remain valuable for data efficiency, control, interpretability, and reliability.

## 13. Essence

The essence of end-to-end deep learning is:

> Train a model to map raw input directly to the final output and optimize the entire trainable system using the final objective.

The basic form is:

$$
\boxed{
x
\xrightarrow{f_\theta}
\hat y
}
$$

Its main advantage is that the model learns useful representations directly from data.

Its main limitation is that learning the complete mapping usually requires a large amount of representative training data.

The key decision question is:

> Do we have enough data to learn the complete input-to-output mapping more effectively than a system that uses explicit intermediate structure?

# Whether to Use End-to-End Deep Learning

## 1. Core Decision

The main question is:

> Should the system learn a direct mapping from raw input to final output, or should it be divided into explicit intermediate modules?

An end-to-end system learns:

$$
x \xrightarrow{f_\theta} \hat y
$$

A modular system uses intermediate stages:

$$
x
\rightarrow
h_1
\rightarrow
h_2
\rightarrow
\cdots
\rightarrow
\hat y
$$

Neither approach is always better. The correct choice depends mainly on the available data, useful intermediate supervision, and system requirements.

## 2. Main Advantages of End-to-End Learning

### Let the Data Learn the Representations

In a modular pipeline, humans decide which intermediate representations should be used.

For example, a traditional speech-recognition system may use:

$$
\text{Audio}
\rightarrow
\text{Acoustic Features}
\rightarrow
\text{Phonemes}
\rightarrow
\text{Words}
\rightarrow
\text{Transcript}
$$

An end-to-end system learns:

$$
\text{Audio}
\rightarrow
\text{Transcript}
$$

The model determines which internal representations are most useful for predicting the final output.

### Optimize the Final Objective Directly

A modular system may optimize several intermediate objectives:

$$
J_1,J_2,\ldots,J_K
$$

These objectives may not align perfectly with the final goal.

An end-to-end system directly optimizes:

$$
J_{\text{final}}
$$

This allows all trainable components to cooperate toward the final output.

## 3. Main Limitation: Large Data Requirements

An end-to-end model must learn the complete mapping:

$$
x\rightarrow y
$$

This includes:

- feature extraction
- intermediate representations
- relationships between representations
- the final decision

Therefore, end-to-end learning often requires a large and representative dataset of final input-output pairs.

For example:

$$
\text{Camera Image}
\rightarrow
\text{Steering Angle}
$$

requires enough driving examples to learn perception, scene understanding, and control behavior from the final supervision alone.

## 4. Value of Intermediate Supervision

A modular system may be preferable when there is limited final input-output data but substantial intermediate labeled data.

For autonomous driving, separate datasets may exist for:

- lane detection
- object detection
- tracking
- trajectory prediction
- path planning

A modular system can use all these sources of supervision:

$$
\text{Image}
\rightarrow
\text{Perception}
\rightarrow
\text{Prediction}
\rightarrow
\text{Planning}
\rightarrow
\text{Control}
$$

A purely end-to-end system may fail to use this intermediate information efficiently.

## 5. Face Recognition Example

A fully end-to-end solution could learn:

$$
(x_1,x_2)
\rightarrow
P(\text{same person})
$$

However, this requires many labeled image pairs.

A more modular solution learns face embeddings:

$$
e_1=f_\theta(x_1)
$$

$$
e_2=f_\theta(x_2)
$$

and compares them:

$$
\hat y
=
\mathbf{1}
\left[
d(e_1,e_2)<\tau
\right]
$$

The embedding approach may be more data-efficient because it can use large identity-classification datasets.

This illustrates an important principle:

> A more direct end-to-end mapping is not automatically the most practical solution.

## 6. When End-to-End Learning Is More Appropriate

End-to-end learning is more attractive when:

1. A large number of labeled final input-output pairs is available.
2. The data represents the production distribution well.
3. The input contains enough information to predict the output.
4. The final objective is clearly defined.
5. Intermediate representations are difficult to design manually.
6. The complete pipeline can be optimized reliably.

In general:

$$
\text{Large final input-output dataset}
\Rightarrow
\text{End-to-end learning becomes more attractive}
$$

## 7. When a Modular System Is More Appropriate

A modular or hybrid system is often preferable when:

- final labeled data is limited
- useful intermediate labels are available
- strong pretrained components already exist
- debugging and interpretation are important
- explicit rules must be enforced
- safety or physical constraints are critical
- some operations are deterministic or non-differentiable
- components must be updated independently

In general:

$$
\text{Limited final data}
+
\text{Useful intermediate supervision}
\Rightarrow
\text{Modular learning becomes more attractive}
$$

## 8. Hybrid Systems

The decision does not have to be completely end-to-end or completely modular.

A practical workflow may be:

1. Train individual modules using intermediate supervision.
2. Connect the modules into a complete pipeline.
3. Fine-tune the trainable components jointly using the final objective.

For example:

$$
x
\rightarrow
f_{\theta_1}
\rightarrow
g_{\theta_2}
\rightarrow
\hat y
$$

The modules may first be pretrained separately and later updated jointly:

$$
\theta_1,\theta_2
\leftarrow
\arg\min J_{\text{final}}
$$

This combines data-efficient modular pretraining with end-to-end optimization.

## 9. Decision Workflow

A useful decision process is:

1. Determine whether enough final input-output data exists.
2. Check whether useful intermediate labels or pretrained modules are available.
3. Evaluate whether manually designed intermediate representations may discard important information.
4. Determine whether interpretability, safety, and control are required.
5. Choose an end-to-end, modular, or hybrid design.

The workflow can be summarized as:

$$
\boxed{
\begin{aligned}
&\text{Enough final data?}\\
&\downarrow\\
&\text{Useful intermediate supervision?}\\
&\downarrow\\
&\text{Need control or interpretability?}\\
&\downarrow\\
&\text{Choose end-to-end, modular, or hybrid}
\end{aligned}
}
$$

## 10. Modern Perspective

Pretrained models reduce the amount of target data required for end-to-end adaptation:

$$
\theta^{(0)}
=
\theta_{\text{pretrained}}
$$

The model can then be fine-tuned on the final task.

However, pretraining does not remove the need for:

- representative target data
- careful evaluation
- error analysis
- domain adaptation
- reliability testing

Many modern AI systems are compound or hybrid systems that combine neural models with:

- search
- retrieval
- planning
- external tools
- explicit rules
- verification components

Therefore, end-to-end learning remains a design choice rather than a universal goal.

## 11. Common Mistakes

### Assuming End-to-End Is Always Better

End-to-end learning may fail when final supervision is insufficient.

### Ignoring Intermediate Data

Intermediate labels can provide valuable supervision and improve data efficiency.

### Designing Overly Rigid Modular Interfaces

Human-designed intermediate representations may discard information needed by downstream modules.

### Evaluating Only Final Accuracy

The design should also consider:

- reliability
- interpretability
- debugging difficulty
- data requirements
- latency
- safety
- maintainability

## 12. Essence

The two main advantages of end-to-end learning are:

1. It allows data to determine useful intermediate representations.
2. It optimizes the complete system directly for the final objective.

The two main limitations are:

1. It often requires a large amount of final input-output data.
2. It may not fully exploit useful intermediate supervision or explicit domain structure.

The central decision rule is:

> Use end-to-end learning when enough representative final data is available to learn the complete mapping reliably.

> Use modular or hybrid learning when intermediate supervision, explicit structure, control, or interpretability provides substantial value.

The correct question is not:

> Is end-to-end better than modular learning?

The correct question is:

> What degree of end-to-end learning is most appropriate for the available data, objectives, and system constraints?